# NB02 — Multimodal Preprocessing (S1 + DEM + IMERG → model-ready samples)

**Goal:** convert the 446 raw hand-labeled Sen1Floods11 chips into aligned, model-ready samples.
Per chip: `X` (6,512,512) = [VV, VH, DEM, rain t−3, rain t−2, rain t−1], `y` (512,512) ∈ {0,1}, `valid_mask` (512,512).
The S1 chip is the **canonical target grid** — never reprojected. Labels never resampled (−1 → invalid).

**Anti-leakage:** no normalization here (physical values stored); norm stats later from TRAIN only. Bolivia never in training. Official splits frozen. Only antecedent rainfall (t−3/t−2/t−1), never event-day/future.

**Caveat:** IMERG is 0.1° while a chip spans ~0.046° — resampled rainfall is smooth antecedent *context*, not fine-scale information.
Read-only raw sources; writes go only to `processed/`.

## 1. Project / context loading
Read the persistent source of truth first (per project rule).

In [ ]:
from pathlib import Path
CTX = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao/project_context")
for f in ["PROJECT_STATE.md", "DECISIONS.md", "DATASET_SPEC.md", "NEXT_STEPS.md"]:
    print("=" * 70)
    print(f"### {f} (head) ###")
    print("\n".join((CTX / f).read_text(encoding="utf-8").splitlines()[:22]))
print("\nCHANGELOG tail:")
print("\n".join((CTX / "CHANGELOG.md").read_text(encoding="utf-8").splitlines()[-12:]))


## 2. Configuration and paths
All paths/config defined once here.

In [ ]:
from pathlib import Path
import datetime
PROJECT_ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
SEN1_ROOT  = Path(r"C:/FloodProject/Sen1Floods11/v1.1")
S1_DIR     = SEN1_ROOT / "data" / "flood_events" / "HandLabeled" / "S1Hand"
LABEL_DIR  = SEN1_ROOT / "data" / "flood_events" / "HandLabeled" / "LabelHand"
SPLIT_DIR  = SEN1_ROOT / "splits" / "flood_handlabeled"
CATALOG_DIR = SEN1_ROOT / "catalog" / "sen1floods11_hand_labeled_source"
DEM_DIR    = Path(r"C:/FloodProject/DEM")
IMERG_DIR  = Path(r"C:/FloodProject/IMERG/data")
OUT_ROOT   = PROJECT_ROOT / "processed"
SPLIT_FILES = {"train": SPLIT_DIR / "flood_train_data.csv", "valid": SPLIT_DIR / "flood_valid_data.csv",
               "test": SPLIT_DIR / "flood_test_data.csv", "bolivia": SPLIT_DIR / "flood_bolivia_data.csv"}
# Preprocessing parameters (documented; physical values stored, no normalization)
DEM_RESAMPLING = "bilinear"   # continuous elevation -> bilinear (D05)
RAIN_RESAMPLING = "bilinear"  # consistency with DEM (D06)
IMERG_BUFFER_DEG = 0.5         # spatial subset buffer around chip before interpolation
DEM_WIN_PAD_PX = 2             # padding for windowed DEM reads (bilinear kernel support)
OVERWRITE_EXISTING = False     # rerun-safe: skip existing .npz unless True
SMOKE_CHIPS = None             # filled in §8 (1 train + 1 valid + 1 bolivia)
for p in [S1_DIR, LABEL_DIR, SPLIT_DIR, CATALOG_DIR, DEM_DIR, IMERG_DIR]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")
print("OUT_ROOT:", OUT_ROOT)


## 3. Dataset / split loading
Official CSVs only (no header). Split assignment is never altered.

In [ ]:
import pandas as pd
split_s1, chip_to_split = {}, {}
for split, csv in SPLIT_FILES.items():
    df = pd.read_csv(csv, header=None, names=["s1_file", "label_file"], dtype=str)
    names = df["s1_file"].str.strip().tolist()
    split_s1[split] = names
    for n in names:
        assert n not in chip_to_split, f"duplicate chip across splits: {n}"
        chip_to_split[n] = split
counts = {k: len(v) for k, v in split_s1.items()}
print(counts, "total =", sum(counts.values()))
assert counts == {"train": 252, "valid": 89, "test": 90, "bolivia": 15}, counts
print("Official splits frozen: OK")


## 4. Source metadata / index preparation
4a — per-chip event dates from the STAC catalog (authoritative; verified one date per event). 4b — DEM tile spatial index built **once** (bounds only). 4c — IMERG filename → date map.

In [ ]:
import json, re
from datetime import date, timedelta
# --- 4a. Event dates from STAC catalog ---
chip_event_date = {}  # s1 filename -> datetime.date (S1 acquisition / flood event day)
for jf in CATALOG_DIR.glob("*/*.json"):
    d = json.loads(jf.read_text())
    chip_event_date[d["id"] + "_S1Hand.tif"] = date.fromisoformat(d["properties"]["datetime"][:10])
print(f"catalog items: {len(chip_event_date)}")
assert len(chip_event_date) == 446, len(chip_event_date)
ev_dates = {}
for s1n, dt in chip_event_date.items():
    ev_dates.setdefault(s1n.split("_")[0], set()).add(dt)
print("event -> dates:", {k: sorted(str(x) for x in v) for k, v in sorted(ev_dates.items())})
assert all(len(v) == 1 for v in ev_dates.values()), "each event must map to exactly one date"
# --- 4b. DEM spatial index (built once) ---
import rasterio
dem_paths = sorted(DEM_DIR.rglob("Copernicus_DSM_COG_10_*_DEM.tif"))
print(f"DEM main tiles: {len(dem_paths)}")
dem_idx = []  # (path, left, bottom, right, top)
for p in dem_paths:
    with rasterio.open(p) as src:
        b = src.bounds
        assert str(src.crs) == "EPSG:4326", (p, src.crs)
        dem_idx.append((p, b.left, b.bottom, b.right, b.top))
print(f"DEM index ready ({len(dem_idx)} tiles, all EPSG:4326). nodata attr check on first tile:", end=" ")
with rasterio.open(dem_idx[0][0]) as s0:
    print(f"nodata={s0.nodata} dtype={s0.dtypes} (None -> non-finite treated as invalid)")
# --- 4c. IMERG date -> file map ---
imerg_by_date = {}
for p in sorted(IMERG_DIR.glob("*.nc4")):
    m = re.search(r"(\d{8})", p.name)
    imerg_by_date[date.fromisoformat(f"{m.group(1)[:4]}-{m.group(1)[4:6]}-{m.group(1)[6:]}")] = p
print(f"IMERG files: {len(imerg_by_date)}")
# verify every chip's t-3/t-2/t-1 files exist (fail fast before any processing)
missing_rain = {}
for s1n, ev in chip_event_date.items():
    need = [ev - timedelta(days=k) for k in (3, 2, 1)]
    if any(d not in imerg_by_date for d in need):
        missing_rain[s1n] = [str(d) for d in need if d not in imerg_by_date]
print("chips missing antecedent rain files:", len(missing_rain))
assert not missing_rain, dict(list(missing_rain.items())[:3])
print("All 446 chips have t-3/t-2/t-1 IMERG coverage. Event-day rainfall is never used.")


## 5. DEM tile lookup / mosaicking utilities
Intersect chip bounds with the cached index → windowed reads (padded) of only intersecting tiles → in-memory mosaic → single bilinear reproject onto the exact S1 grid.

In [ ]:
import numpy as np
import rasterio
from rasterio import windows
from rasterio.io import MemoryFile
from rasterio.merge import merge as rio_merge
from rasterio.warp import reproject, Resampling

def dem_tiles_for(bounds):
    """Return cached DEM tile paths intersecting (left, bottom, right, top)."""
    l, b, r, t = bounds
    return [p for (p, pl, pb, pr, pt) in dem_idx if pl < r and pr > l and pb < t and pt > b]

def dem_on_s1_grid(s1_profile, chip_bounds, tile_paths):
    """Reproject mosaicked DEM onto the exact S1 grid (bilinear, metres, NaN = void).
    Windowed tile reads keep memory bounded; supports 1..N intersecting tiles."""
    l, b, r, t = chip_bounds
    mems, srcs = [], []
    try:
        for tp in tile_paths:
            with rasterio.open(tp) as src:
                win = windows.from_bounds(l, b, r, t, transform=src.transform)
                win = win.round_offsets().round_lengths()
                win = windows.Window(win.col_off - DEM_WIN_PAD_PX, win.row_off - DEM_WIN_PAD_PX,
                                     win.width + 2 * DEM_WIN_PAD_PX, win.height + 2 * DEM_WIN_PAD_PX)
                arr = src.read(1, window=win, boundless=True, fill_value=np.nan)
                wtransform = windows.transform(win, src.transform)
                mf = MemoryFile()
                mems.append(mf)
                ds = mf.open(driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                             dtype="float32", crs="EPSG:4326", transform=wtransform, nodata=np.nan)
                ds.write(arr.astype(np.float32), 1)
                srcs.append(ds)
        mosaic, mosaic_transform = rio_merge(srcs, nodata=np.nan)
        dst = np.full((s1_profile["height"], s1_profile["width"]), np.nan, dtype=np.float32)
        reproject(mosaic, dst, src_transform=mosaic_transform,
                  src_crs="EPSG:4326", dst_transform=s1_profile["transform"], dst_crs=s1_profile["crs"],
                  resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
        return dst
    finally:
        for s in srcs:
            s.close()
        for m in mems:
            m.close()


## 6. IMERG loading / interpolation utilities
`precipitation` only. Explicit longitude/latitude coordinates (orientation detected at runtime — never assumed). Spatial subset around the chip first, then vectorized bilinear interpolation onto the exact S1 pixel grid.

In [ ]:
import numpy as np
import xarray as xr

def load_imerg_subset(nc_path, bounds, buffer_deg=IMERG_BUFFER_DEG):
    """Load `precipitation` (time=0) subset around bounds. Returns (lon1d asc, lat1d asc, vals[nlat,nlon])."""
    l, b, r, t = bounds
    with xr.open_dataset(nc_path) as ds:
        assert "precipitation" in ds.data_vars, f"no precipitation in {nc_path}"
        lon = np.asarray(ds["lon"].values, dtype=np.float64)
        lat = np.asarray(ds["lat"].values, dtype=np.float64)
        # explicit orientation handling: sort to ascending, reorder values identically
        io = np.argsort(lon); lon = lon[io]
        ia = np.argsort(lat); lat = lat[ia]
        da = ds["precipitation"].isel(time=0).transpose("lat", "lon")  # explicit dim order: never assume axis layout
        vals = np.asarray(da.values, dtype=np.float64)
        vals = vals[ia][:, io]
        mlo = (lon >= l - buffer_deg) & (lon <= r + buffer_deg)
        mla = (lat >= b - buffer_deg) & (lat <= t + buffer_deg)
        assert mlo.sum() >= 2 and mla.sum() >= 2, f"subset too small for {nc_path} bounds={bounds}"
        return lon[mlo], lat[mla], vals[np.ix_(mla, mlo)]

def bilinear_on_grid(src_lon, src_lat, src_vals, tgt_lon, tgt_lat):
    """Vectorized bilinear interpolation. src axes ascending; out-of-range -> NaN."""
    assert np.all(np.diff(src_lon) > 0) and np.all(np.diff(src_lat) > 0)
    j = np.clip(np.searchsorted(src_lon, tgt_lon, side="right") - 1, 0, src_lon.size - 2)
    i = np.clip(np.searchsorted(src_lat, tgt_lat, side="right") - 1, 0, src_lat.size - 2)
    x0, x1 = src_lon[j], src_lon[j + 1]
    y0, y1 = src_lat[i], src_lat[i + 1]
    fx = np.where(x1 > x0, (tgt_lon - x0) / np.where(x1 == x0, 1, x1 - x0), 0.0)
    fy = np.where(y1 > y0, (tgt_lat - y0) / np.where(y1 == y0, 1, y1 - y0), 0.0)
    v00, v10 = src_vals[i, j], src_vals[i, j + 1]
    v01, v11 = src_vals[i + 1, j], src_vals[i + 1, j + 1]
    out = (1 - fx) * (1 - fy) * v00 + fx * (1 - fy) * v10 + (1 - fx) * fy * v01 + fx * fy * v11
    bad = (tgt_lon < src_lon[0]) | (tgt_lon > src_lon[-1]) | (tgt_lat < src_lat[0]) | (tgt_lat > src_lat[-1])
    return np.where(bad | ~np.isfinite(out), np.nan, out)

def s1_grid_lonlat(transform, height, width):
    """Pixel-center lon/lat of the S1 grid (exact canonical grid)."""
    cc, rr = np.meshgrid(np.arange(width, dtype=np.float64) + 0.5, np.arange(height, dtype=np.float64) + 0.5)
    lon = transform.c + transform.a * cc + transform.b * rr
    lat = transform.f + transform.d * cc + transform.e * rr
    return lon, lat


## 7. Single-chip preprocessing function
Reads S1 + label (aligned, no resampling), DEM (mosaic + bilinear), 3 antecedent rain days (bilinear). Tracks validity; fills with documented neutral values so `X` is finite; `valid_mask` excludes invalid pixels from loss.

In [ ]:
import numpy as np
import rasterio
from datetime import timedelta

FILL_FALLBACK = 0.0  # used only if a channel has zero valid pixels

def preprocess_chip(s1_name, imerg_cache=None):
    """Process one chip -> dict with X(6,512,512) f32, y(512,512) u8, valid_mask bool + metadata.
    imerg_cache: optional dict date->(lon,lat,vals) full-grid arrays to avoid re-reading files."""
    split = chip_to_split[s1_name]
    event = s1_name.split("_")[0]
    ev_date = chip_event_date[s1_name]
    rain_dates = [ev_date - timedelta(days=k) for k in (3, 2, 1)]
    assert all(d < ev_date for d in rain_dates), "only antecedent rainfall allowed"
    # --- S1 (canonical grid; NaNs tracked, never treated as 0 dB) ---
    with rasterio.open(S1_DIR / s1_name) as src:
        assert src.count == 2 and (src.height, src.width) == (512, 512), (s1_name, src.count, src.shape)
        assert str(src.crs) == "EPSG:4326", src.crs
        s1 = src.read().astype(np.float32)  # (2,512,512) float32 dB
        transform, crs, bounds = src.transform, src.crs, src.bounds
    prof = {"height": 512, "width": 512, "transform": transform, "crs": crs}
    cb = (bounds.left, bounds.bottom, bounds.right, bounds.top)
    vv, vh = s1[0], s1[1]
    s1_valid = np.isfinite(vv) & np.isfinite(vh)
    vv_f = np.where(np.isfinite(vv), vv, np.nanmean(vv) if np.isfinite(vv).any() else FILL_FALLBACK).astype(np.float32)
    vh_f = np.where(np.isfinite(vh), vh, np.nanmean(vh) if np.isfinite(vh).any() else FILL_FALLBACK).astype(np.float32)
    # --- Label (aligned: no resample; -1 -> invalid) ---
    with rasterio.open(LABEL_DIR / s1_name.replace("_S1Hand.tif", "_LabelHand.tif")) as src:
        assert (src.height, src.width) == (512, 512), src.shape
        lab = src.read(1)
    assert set(np.unique(lab).tolist()) <= {-1, 0, 1}, np.unique(lab)
    label_valid = lab != -1
    y = np.where(lab == 1, 1, 0).astype(np.uint8)
    # --- DEM (mosaic intersecting tiles -> bilinear onto S1 grid) ---
    tiles = dem_tiles_for(cb)
    assert tiles, f"no DEM coverage for {s1_name} bounds={cb}"
    dem = dem_on_s1_grid(prof, cb, tiles)
    dem_valid = np.isfinite(dem)
    dem_med = float(np.nanmedian(dem)) if dem_valid.any() else FILL_FALLBACK
    dem_f = np.where(dem_valid, dem, dem_med).astype(np.float32)
    # --- IMERG t-3/t-2/t-1 (subset -> bilinear; explicit coords) ---
    glon, glat = s1_grid_lonlat(transform, 512, 512)
    rains, rain_valids = [], []
    for d in rain_dates:
        if imerg_cache is not None and d in imerg_cache:
            lon, lat, vals = imerg_cache[d]
        else:
            lon, lat, vals = load_imerg_subset(imerg_by_date[d], cb)
        r = bilinear_on_grid(lon, lat, vals, glon, glat).astype(np.float32)
        rains.append(np.where(np.isfinite(r), r, 0.0).astype(np.float32))  # missing rain -> 0 mm/day, masked invalid
        rain_valids.append(np.isfinite(r))
    rain_valid = np.logical_and.reduce(rain_valids)
    # --- Assemble ---
    X = np.stack([vv_f, vh_f, dem_f] + rains, axis=0)  # (6,512,512) float32
    assert X.shape == (6, 512, 512) and X.dtype == np.float32
    assert all(np.isfinite(X).reshape(6, -1).all(axis=1)), f"non-finite values remain in {s1_name}"
    valid_mask = label_valid & s1_valid & dem_valid & rain_valid
    y = np.where(valid_mask, y, 0).astype(np.uint8)  # ignore-index: masked px read 0, excluded via valid_mask
    meta = {"chip_id": s1_name.replace("_S1Hand.tif", ""), "s1_file": s1_name, "split": split,
            "event": event, "event_date": ev_date.isoformat(),
            "rain_t3": rain_dates[0].isoformat(), "rain_t2": rain_dates[1].isoformat(),
            "rain_t1": rain_dates[2].isoformat(), "dem_tiles": ";".join(t.parent.name for t in tiles),
            "n_dem_tiles": len(tiles), "s1_nan_pct": round(100 * float((~s1_valid).sum()) / s1_valid.size, 4),
            "flood_pct_valid": round(100 * float(((lab == 1) & label_valid).sum()) / max(int(label_valid.sum()), 1), 4),
            "valid_frac": round(float(valid_mask.sum()) / valid_mask.size, 6)}
    return {"X": X, "y": y, "valid_mask": valid_mask, "meta": meta,
            "profile": {"crs": str(crs), "transform": tuple(transform), "bounds": cb}}
print("preprocess_chip defined. Fill strategy: S1 NaN -> per-chip valid mean; DEM void -> chip median; rain NaN -> 0.0.")
print("All filled pixels remain excluded via valid_mask (except rain==0 genuine zeros, which are valid).")


## 8. Three-sample smoke test
Exactly 3 chips (1 train + 1 valid + 1 Bolivia) before the full run. Automated alignment/shape/date assertions must pass.

In [ ]:
SMOKE_CHIPS = [sorted(split_s1["train"])[0], sorted(split_s1["valid"])[0], sorted(split_s1["bolivia"])[0]]
print("smoke chips:", SMOKE_CHIPS)
smoke = {}
for n in SMOKE_CHIPS:
    r = preprocess_chip(n)
    m = r["meta"]
    assert r["X"].shape == (6, 512, 512) and r["X"].dtype == np.float32
    assert r["y"].shape == (512, 512) and r["valid_mask"].shape == (512, 512)
    assert set(np.unique(r["y"][r["valid_mask"]]).tolist()) <= {0, 1}
    assert np.isfinite(r["X"]).all()
    assert m["rain_t1"] < m["event_date"] and m["rain_t3"] < m["rain_t2"] < m["rain_t1"]
    smoke[n] = r
    print(f"{n}: split={m['split']} event={m['event']} event_date={m['event_date']} "
          f"rains={m['rain_t3']},{m['rain_t2']},{m['rain_t1']} dem_tiles={m['n_dem_tiles']} "
          f"s1_nan%={m['s1_nan_pct']} flood%={m['flood_pct_valid']} valid_frac={m['valid_frac']}")
print("SMOKE TEST: ALL ASSERTIONS PASS")


## 9. Visual QC
Per smoke chip: VV | VH | DEM | rain t−3 | rain t−2 | rain t−1 | label (+ valid-mask overlay outline); plus channel stats. Confirm spatial co-registration by eye before §10.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
label_cmap = ListedColormap(["#808080", "#1a1a1a", "#00b3ff"])  # -1 grey, 0 black, 1 blue
for n, r in smoke.items():
    X, y, vm, m = r["X"], r["y"], r["valid_mask"], r["meta"]
    lab_show = np.where(vm, y, -1)
    titles = [f"VV dB [{np.min(X[0]):.1f},{np.max(X[0]):.1f}] m={np.mean(X[0]):.1f}",
              f"VH dB [{np.min(X[1]):.1f},{np.max(X[1]):.1f}] m={np.mean(X[1]):.1f}",
              f"DEM m [{np.min(X[2]):.1f},{np.max(X[2]):.1f}] m={np.mean(X[2]):.1f}",
              f"rain t-3 {m['rain_t3']} max={np.max(X[3]):.2f}",
              f"rain t-2 {m['rain_t2']} max={np.max(X[4]):.2f}",
              f"rain t-1 {m['rain_t1']} max={np.max(X[5]):.2f}",
              f"label flood={m['flood_pct_valid']}% (grey=invalid)"]
    fig, ax = plt.subplots(2, 4, figsize=(15, 7))
    ax = ax.ravel()
    for i in range(6):
        v0, v1 = np.percentile(X[i][np.isfinite(X[i])], [2, 98]) if i < 3 else (np.min(X[i]), np.max(X[i]))
        ax[i].imshow(X[i], cmap="gray" if i < 2 else ("terrain" if i == 2 else "Blues"),
                     vmin=v0 if v1 > v0 else 0, vmax=v1 if v1 > v0 else 1)
        ax[i].set_title(titles[i], fontsize=9)
    ax[6].imshow(lab_show, cmap=label_cmap, vmin=-1, vmax=1, interpolation="nearest")
    ax[6].set_title(titles[6], fontsize=9)
    ax[7].imshow(vm, cmap="gray")
    ax[7].set_title(f"valid_mask True={vm.mean()*100:.1f}% (black=excluded)", fontsize=9)
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{n} | {m['split']} | {m['event']} event {m['event_date']} | CRS={r['profile']['crs']} grid=512x512", fontsize=10)
    fig.tight_layout()
    plt.show()
    del lab_show
print("QC note: rainfall panels look near-flat because one chip (~0.046 deg) is smaller than one IMERG cell (0.1 deg) — expected (see caveat).")


## 10. Full preprocessing
One chip at a time (never the whole dataset in RAM). IMERG full-date arrays cached per event (3 × ~26 MB transient) to avoid re-reading global files per chip. Rerun-safe: existing `.npz` files are skipped unless `OVERWRITE_EXISTING=True`.

In [ ]:
import time, gc
from collections import defaultdict
OUT_ROOT.mkdir(parents=True, exist_ok=True)
for s in ["train", "valid", "test", "bolivia"]:
    (OUT_ROOT / s).mkdir(parents=True, exist_ok=True)

def save_sample(res):
    m = res["meta"]
    out = OUT_ROOT / m["split"] / (m["chip_id"] + ".npz")
    if out.exists() and not OVERWRITE_EXISTING:
        return out, True
    np.savez_compressed(out, X=res["X"], y=res["y"], valid_mask=res["valid_mask"],
                        chip_id=m["chip_id"], s1_file=m["s1_file"], split=m["split"], event=m["event"],
                        event_date=m["event_date"], rain_t3=m["rain_t3"], rain_t2=m["rain_t2"], rain_t1=m["rain_t1"],
                        dem_tiles=m["dem_tiles"], crs=res["profile"]["crs"],
                        transform=np.array(res["profile"]["transform"], dtype=np.float64),
                        bounds=np.array(res["profile"]["bounds"], dtype=np.float64))
    return out, False

by_event = defaultdict(list)
for s1n in chip_to_split:
    by_event[s1n.split("_")[0]].append(s1n)
print(f"events: {sorted(by_event)} | total chips: {sum(len(v) for v in by_event.values())}")
records, t_all, n_skip = [], time.time(), 0
for ev in sorted(by_event):
    names = sorted(by_event[ev])
    ev_date = chip_event_date[names[0]]
    # per-event IMERG cache: 3 antecedent dates, full-grid arrays loaded once
    cache, t0 = {}, time.time()
    for d in [ev_date - timedelta(days=k) for k in (3, 2, 1)]:
        with xr.open_dataset(imerg_by_date[d]) as ds:
            lon = np.asarray(ds["lon"].values, dtype=np.float64); lat = np.asarray(ds["lat"].values, dtype=np.float64)
            io, ia = np.argsort(lon), np.argsort(lat)
            da = ds["precipitation"].isel(time=0).transpose("lat", "lon")
            cache[d] = (lon[io], lat[ia], np.asarray(da.values, dtype=np.float64)[ia][:, io])
    for j, s1n in enumerate(names):
        res = preprocess_chip(s1n, imerg_cache=cache)
        out, skipped = save_sample(res)
        n_skip += skipped
        records.append({**res["meta"], "path": str(out.relative_to(OUT_ROOT))})
        del res
        if (j + 1) % 25 == 0 or j + 1 == len(names):
            print(f"  {ev}: {j+1}/{len(names)} (skipped_existing={n_skip})", flush=True)
    del cache
    gc.collect()
    print(f"[{ev}] {len(names)} chips in {time.time()-t0:.1f}s", flush=True)
print(f"\nFULL RUN: {len(records)} chips in {time.time()-t_all:.1f}s, skipped(existing)={n_skip}")
rec_df = pd.DataFrame(records)
rec_df.to_csv(OUT_ROOT / "metadata.csv", index=False)
print("metadata.csv rows:", len(rec_df))
display(rec_df.head(3))


## 11. Final dataset validation
Reload every `.npz` (one at a time) and verify counts, shapes, dtypes, label/rain/grid integrity.

In [ ]:
import numpy as np
checks = []
def check(name, cond, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(cond) else "FAIL", "detail": detail})
seen, per_split = set(), {s: 0 for s in ["train", "valid", "test", "bolivia"]}
bad = {"shape": [], "dtype": [], "yval": [], "nonfinite": [], "rain": [], "split": [], "dem": []}
for _, row in rec_df.iterrows():
    p = OUT_ROOT / row["path"]
    d = np.load(p, allow_pickle=True)
    X, y, vm = d["X"], d["y"], d["valid_mask"]
    key = str(d["chip_id"])
    if key in seen:
        bad["split"].append(key + " DUP")
    seen.add(key)
    if X.shape != (6, 512, 512): bad["shape"].append(key)
    if X.dtype != np.float32: bad["dtype"].append(key)
    if y.shape != (512, 512) or vm.shape != (512, 512): bad["shape"].append(key + " y/vm")
    if not set(np.unique(y[vm]).tolist()) <= {0, 1}: bad["yval"].append(key)
    if (y[~vm] != 0).any(): bad["yval"].append(key + " invalid!=0")
    if not np.isfinite(X).all(): bad["nonfinite"].append(key)
    if not np.isfinite(X[2]).all() or (X[2][vm].size and not np.isfinite(X[2][vm]).any()): bad["dem"].append(key)
    exp_rain = [(chip_event_date[row['s1_file']] - timedelta(days=k)).isoformat() for k in (3, 2, 1)]
    if [str(d["rain_t3"]), str(d["rain_t2"]), str(d["rain_t1"])] != exp_rain: bad["rain"].append(key)
    if str(d["split"]) != row["split"] or str(d["split"]) != chip_to_split[row["s1_file"]]: bad["split"].append(key)
    per_split[row["split"]] += 1
check("train==252", per_split["train"] == 252, str(per_split))
check("valid==89", per_split["valid"] == 89, str(per_split))
check("test==90", per_split["test"] == 90, str(per_split))
check("bolivia==15", per_split["bolivia"] == 15, str(per_split))
check("total==446, no duplicates", len(seen) == 446 == len(rec_df), f"seen={len(seen)} rows={len(rec_df)}")
check("every X (6,512,512)", not bad["shape"], str(bad["shape"][:3]))
check("X dtype float32", not bad["dtype"], str(bad["dtype"][:3]))
check("y valid {0,1}, invalid==0", not bad["yval"], str(bad["yval"][:3]))
check("X fully finite", not bad["nonfinite"], str(bad["nonfinite"][:3]))
check("DEM valid coverage", not bad["dem"], str(bad["dem"][:3]))
check("rain dates == event-3/-2/-1", not bad["rain"], str(bad["rain"][:3]))
check("split dirs match official splits", not bad["split"], str(bad["split"][:3]))
# grid alignment: transform/bounds stored per sample equal S1 source (sampled audit)
import rasterio as _rio
mis = []
for _, row in rec_df.iloc[::50].iterrows():
    d = np.load(OUT_ROOT / row["path"], allow_pickle=True)
    with _rio.open(S1_DIR / row["s1_file"]) as src:
        if tuple(d["transform"]) != tuple(src.transform) or tuple(d["bounds"]) != tuple(src.bounds):
            mis.append(row["s1_file"])
check("stored grid == S1 canonical grid (sampled)", not mis, str(mis[:3]))
cdf = pd.DataFrame(checks)
display(cdf)
print("VALIDATION:", "ALL PASS" if (cdf.status == "PASS").all() else "FAILURES — see rows")


## 12. Summary / results
Compact report + `dataset_summary.json`. No normalization statistics are computed here by design.

In [ ]:
import json
summary = {
    "notebook": "NB02_Multimodal_Preprocessing", "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "counts": per_split, "total": int(len(rec_df)),
    "channels": ["VV_dB", "VH_dB", "DEM_m", "rain_t-3_mm/day", "rain_t-2_mm/day", "rain_t-1_mm/day"],
    "shapes": {"X": [6, 512, 512], "y": [512, 512], "valid_mask": [512, 512]},
    "dtypes": {"X": "float32", "y": "uint8", "valid_mask": "bool"},
    "interpolation": {"DEM": DEM_RESAMPLING, "IMERG": RAIN_RESAMPLING},
    "fill_strategy": "S1 NaN->per-chip valid mean; DEM void->chip median; rain NaN->0.0; all tracked in valid_mask",
    "normalization": "NONE in NB02 (physical values); train-only stats deferred to NB03",
    "mean_valid_frac": round(float(rec_df["valid_frac"].mean()), 6),
    "mean_s1_nan_pct": round(float(rec_df["s1_nan_pct"].mean()), 4),
    "multi_tile_chips": int((rec_df["n_dem_tiles"] > 1).sum()),
    "overwrite_existing": OVERWRITE_EXISTING,
    "validation": ("ALL PASS" if (cdf.status == "PASS").all() else "FAILURES"),
}
(OUT_ROOT / "dataset_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nPROCESSED DATASET COMPLETE:", f"{summary['total']} samples", per_split)
print("README for NB03: X holds physical values (dB / metres / mm-day). Compute normalization from TRAIN ONLY.")


## 13. Context / changelog update
Timestamped source-of-truth update (rule: every meaningful step updates context).

In [ ]:
import datetime
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
locked = (summary["validation"] == "ALL PASS")
entry = (f"\n## {now} - NB02_Multimodal_Preprocessing {'COMPLETE' if locked else 'RUN WITH FAILURES'}\n"
         f"- Processed {summary['total']} chips {summary['counts']} -> processed/{{train,valid,test,bolivia}}/*.npz\n"
         f"- Format: X(6,512,512) f32 [VV,VH,DEM,rain t-3/t-2/t-1], y(512,512) u8, valid_mask bool + metadata.csv/dataset_summary.json\n"
         f"- DEM: cached index, windowed mosaic, bilinear; multi-tile chips: {summary['multi_tile_chips']}\n"
         f"- IMERG: precipitation t-3/t-2/t-1 from STAC dates, explicit coords, bilinear; mean_valid_frac={summary['mean_valid_frac']}\n"
         f"- Fill: S1 NaN->chip mean, DEM void->chip median, rain NaN->0.0; y zeroed where ~valid_mask (ignore-index)\n"
         f"- No normalization (train-only stats -> NB03). Validation: {summary['validation']}. "
         f"{'NB02 safe to lock.' if locked else 'DO NOT LOCK - fix failures first.'}\n")
cl = CTX / "CHANGELOG.md"
cl.write_text(cl.read_text(encoding="utf-8") + entry, encoding="utf-8")
ps = CTX / "PROJECT_STATE.md"
t = ps.read_text(encoding="utf-8")
import re
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (NB02 complete: " + ("yes" if locked else "NO - validation failed") + ")", t, count=1, flags=re.M)
t += f"\n\n## NB02 result ({now})\n- {summary['total']} processed samples {summary['counts']}; validation {summary['validation']}; mean_valid_frac={summary['mean_valid_frac']}.\n"
ps.write_text(t, encoding="utf-8")
nx = CTX / "NEXT_STEPS.md"
t = nx.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now}", t, count=1, flags=re.M)
nx.write_text(t + (f"\n\n## Update ({now})\n- NB02 COMPLETE + validated. Next: NB03 train-only normalization + dataloaders.\n" if locked else f"\n\n## Update ({now})\n- NB02 validation FAILED - fix before NB03.\n"), encoding="utf-8")
print(entry)
print("Context files updated with timestamp:", now)
